# Enterprise MCP Proxy

This notebook demonstrates the MCP proxy -- a single entry point that fronts multiple backend MCP servers, adding authorization, rate limiting, circuit breaking, audit, and accounting.

Start two backend MCP servers and the proxy in separate terminals before running this notebook:
```bash
fastmcp run example_mcp_server.py --port 8301
fastmcp run example_mcp_server_v2.py --port 8302
fastmcp run example_mcp_proxy_server.py --port 8200
```

In [ ]:
from fastmcp import Client

PROXY_URL = "http://127.0.0.1:8200/mcp/"

## 1. Server Composition

The proxy mounts both backend servers using `FastMCP.as_proxy()` + `mount()`. Tools are automatically namespaced with the backend name as prefix. Prompts and resources from backends are also forwarded.

In [ ]:
async with Client(PROXY_URL) as client:
    tools = await client.list_tools()
    print(f"Available tools ({len(tools)}):")
    for tool in tools:
        print(f"  {tool.name}: {tool.description}")

## 2. Authorization

The proxy enforces role-based access policies using glob patterns. The example configuration allows the `default` role to call `demo_*` tools but denies `demo_v2_*`. Deny rules take precedence over allow rules. When no rule matches, access is denied by default.

In [ ]:
# Allowed: default role can call demo_* tools
async with Client(PROXY_URL) as client:
    result = await client.call_tool("demo_add", {"a": 40123456789, "b": 2123456789})
    print(f"demo_add result: {result}")

In [ ]:
# Denied: default role cannot call demo_v2_* tools
async with Client(PROXY_URL) as client:
    try:
        result = await client.call_tool("demo_v2_add", {"a": 1, "b": 2})
        print(f"demo_v2_add result: {result}")
    except Exception as e:
        print(f"Authorization denied (expected): {e}")

## 3. Rate Limiting

The proxy uses FastMCP's built-in `RateLimitingMiddleware` with configurable requests-per-second and burst capacity. The example is configured at 5 req/s with burst capacity of 10. Sending a burst of requests triggers rate limiting.

In [ ]:
# Rapid-fire calls to trigger rate limiting
async with Client(PROXY_URL) as client:
    for i in range(15):
        try:
            result = await client.call_tool("demo_add", {"a": i, "b": 1})
            print(f"  Call {i+1}: OK")
        except Exception as e:
            print(f"  Call {i+1}: Rate limited - {e}")
            break

## 4. Audit Trail

The `AuditMiddleware` records every tool invocation to an append-only JSON Lines file with full context: timestamp, user, tenant, session, tool, arguments, status, and duration. The proxy exposes this as an introspection tool.

In [ ]:
import json

async with Client(PROXY_URL) as client:
    result = await client.call_tool("proxy_audit_read", {"limit": 5})
    entries = json.loads(result[0].text)
    print(f"Recent audit entries ({len(entries)}):")
    for entry in entries:
        print(f"  {entry['tool']} -> {entry['status']} ({entry['duration_ms']:.0f}ms)")

## 5. Accounting

The `AccountingMiddleware` tracks usage per user, tenant, session, and tool. Budget alerts fire at configurable thresholds. The proxy exposes usage statistics as an introspection tool.

In [ ]:
async with Client(PROXY_URL) as client:
    result = await client.call_tool("proxy_accounting_usage", {})
    usage = json.loads(result[0].text)
    print(json.dumps(usage, indent=2))

## 6. Other Concerns

**Authentication.** The `AuthSessionMiddleware` (reused from `core/mcp/`) validates JWT tokens at entry and sets the user session context. Without a token, the user defaults to `default`. In production, agents present JWTs to identify themselves.

**Multi-tenancy.** Tenant identity is extracted from JWT claims (`org_id`). Each tenant gets isolated audit entries and budget tracking. The accounting output above shows per-tenant usage -- in a multi-tenant setup, different organizations would see separate counters.

**Circuit breaking.** The `CircuitBreakerMiddleware` tracks consecutive failures per backend. After 5 failures (configurable), the circuit opens and requests are rejected immediately. After a 30-second recovery timeout, the circuit enters half-open state to probe recovery. To see this in action, stop one of the backend servers and call its tools.

**Network isolation.** Handled at the backend level using `MCPServerPrivateData` (dual-instance routing). The proxy delegates to backends that already enforce isolation, keeping the proxy focused on cross-cutting concerns.

**Observability.** FastMCP's `LoggingMiddleware` emits structured logs for every operation. Check the proxy's terminal output to see request/response logging alongside the audit middleware's persistent records.